# Week 10 – Build a Simple Retrieval-Augmented Generation (RAG) Assistant

**Level:** Beginner  
**Environment:** Google Colab  
**Tools:** Python, scikit-learn, Gemini API  
**Project goal:** Build a small assistant that answers questions using five approved sample documents, shows the retrieved source passages, and honestly reports when the documents do not contain the answer.

## What is RAG?

Retrieval-Augmented Generation (RAG) combines two steps:

1. **Retrieval:** Find relevant information from a collection of documents.
2. **Generation:** Give that information to an AI model so it can produce an answer grounded in the documents.

Our simple workflow:

`Documents → Chunks → TF-IDF vectors → Similarity search → Top relevant chunks → Gemini → Answer with sources`

## What this notebook will cover

1. Install and import libraries.
2. Create and load five sample documents.
3. Split documents into character-based chunks.
4. Convert chunks into TF-IDF vectors.
5. Compare a question with chunks using cosine similarity.
6. Retrieve the top relevant chunks.
7. Discover and test Gemini models available to the API key.
8. Generate answers using retrieved context.
9. Display source passages and handle unsupported questions.
10. Evaluate the assistant using five questions.

**Important:** This is a deliberately simple learning project. TF-IDF is keyword-based retrieval, not a deep semantic search engine. The notebook explains this limitation where relevant.

## 1. Install the required libraries

We use:
- `scikit-learn` for TF-IDF and cosine similarity.
- `google-genai` for communicating with Gemini.
- `requests` for listing models available to our API key.

Run this cell once in the Colab runtime. If Colab asks you to restart the runtime after installation, do so and then run the notebook cells from the beginning.

In [1]:
!pip install -q scikit-learn google-genai requests

### Cell explanation and output analysis

- `pip install` downloads the Python packages needed by this notebook.
- `-q` means quiet mode, so installation logs are reduced.
- A successful installation usually finishes without a Python error. The cell may show little or no output.

## 2. Import libraries

A library is a collection of reusable code. We import only the functions needed for this project.

In [2]:
import os
import time
import requests
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from google import genai

print("All libraries imported successfully!")

All libraries imported successfully!


### Cell explanation

- `numpy`: Helps us sort similarity scores and work with numerical arrays.
- `TfidfVectorizer`: Converts text into TF-IDF numerical vectors.
- `cosine_similarity`: Measures similarity between the question vector and document vectors.
- `genai`: Provides the Gemini client.
- `requests`: Calls the Gemini model-list endpoint.
- `time`: Adds a short pause between model tests if needed.

**Expected output:** `All libraries imported successfully!`

## 3. Create the sample knowledge base

A knowledge base is the collection of information our RAG assistant is allowed to use. To keep this assignment simple, we create five short text documents directly in Python instead of uploading PDFs.

The dictionary key is the filename. The value is the document text.

In [3]:
documents = {
    "python.txt": """
    Python is a high-level programming language.
    It is easy to learn and widely used in data science,
    artificial intelligence, web development and automation.
    Python supports object-oriented and procedural programming.
    """,

    "java.txt": """
    Java is a popular object-oriented programming language.
    It is widely used for enterprise applications,
    Android development and backend systems.
    Java runs on the Java Virtual Machine (JVM).
    """,

    "sql.txt": """
    SQL stands for Structured Query Language.
    It is used to store, retrieve, update and delete data
    in relational database management systems.
    SQL supports commands such as SELECT, INSERT,
    UPDATE and DELETE.
    """,

    "machine_learning.txt": """
    Machine Learning is a branch of Artificial Intelligence.
    It allows computers to learn patterns from data.
    Supervised learning uses labelled training data.
    Unsupervised learning finds patterns in unlabelled data.
    """,

    "deep_learning.txt": """
    Deep Learning is a subset of Machine Learning.
    It uses artificial neural networks with multiple layers.
    Deep learning is used in image recognition,
    speech recognition and natural language processing.
    """
}

print("Total documents:", len(documents))

for filename in documents:
    print("-", filename)

Total documents: 5
- python.txt
- java.txt
- sql.txt
- machine_learning.txt
- deep_learning.txt


### Cell explanation

- `documents` is a Python dictionary containing five documents.
- Each filename is a key, such as `python.txt`.
- Each multiline string is the content of that document.
- `len(documents)` counts how many documents we created.
- The loop prints the filenames.

**Expected output:** Five filenames. These are our approved source documents for the assignment.

## 4. Load the documents

Document loading means reading document contents into a structure that the rest of the program can process. Our documents are already in memory, so we copy each filename and its cleaned text into a list of records.

We keep the source filename because we will need it for citations.

In [4]:
loaded_documents = []

for filename, content in documents.items():
    loaded_documents.append({
        "source": filename,
        "content": content.strip()
    })

print("Total loaded documents:", len(loaded_documents))

for doc in loaded_documents:
    print("\nSource:", doc["source"])
    print("Content:", doc["content"])

Total loaded documents: 5

Source: python.txt
Content: Python is a high-level programming language.
    It is easy to learn and widely used in data science,
    artificial intelligence, web development and automation.
    Python supports object-oriented and procedural programming.

Source: java.txt
Content: Java is a popular object-oriented programming language.
    It is widely used for enterprise applications,
    Android development and backend systems.
    Java runs on the Java Virtual Machine (JVM).

Source: sql.txt
Content: SQL stands for Structured Query Language.
    It is used to store, retrieve, update and delete data
    in relational database management systems.
    SQL supports commands such as SELECT, INSERT,
    UPDATE and DELETE.

Source: machine_learning.txt
Content: Machine Learning is a branch of Artificial Intelligence.
    It allows computers to learn patterns from data.
    Supervised learning uses labelled training data.
    Unsupervised learning finds patterns i

### Cell explanation and output analysis

- `loaded_documents = []` creates an empty list.
- `.items()` gives us each filename and its content.
- `.strip()` removes whitespace at the beginning and end of the text.
- Each list item is a dictionary with `source` and `content`.

**Expected output:** Five loaded records, each showing a filename and its text. In a real project, the same loading step could read text extracted from PDFs, Word files, or text files.

## 5. Understand document chunking

A **chunk** is a smaller piece of a document. We split documents because retrieving a few relevant passages is usually more useful than sending every full document to the language model.

This notebook uses **character-based chunking**:
- `chunk_size=100`: each chunk contains up to 100 characters.
- `overlap=20`: the next chunk starts 20 characters before the previous chunk ends.

The next starting position is:

`next_start = current_start + chunk_size - overlap`

For example, with a chunk size of 100 and overlap of 20, the next chunk starts 80 characters after the previous chunk started.

This simple method splits by character position. It can cut a word or sentence in the middle. We keep it because it is easy to understand; more advanced splitters can preserve sentence or paragraph boundaries.

### Visual example

Suppose the text is represented by character positions:

- Chunk 1: positions 0–99
- Chunk 2: positions 80–179
- Chunk 3: positions 160–259

The 20 characters at the end of one chunk are repeated at the beginning of the next chunk. This repeated region is called **overlap**.

Overlap can help preserve context around chunk boundaries. It also means some text appears in more than one chunk.

## 6. Create the character-based chunking function

The function receives a text string, takes a slice of up to `chunk_size` characters, stores it, and moves forward by `chunk_size - overlap` characters.

In [5]:
def split_text(text, chunk_size=100, overlap=20):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than zero")

    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be at least zero and smaller than chunk_size")

    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)

        start = end - overlap

    return chunks


print("Chunking function created successfully!")

Chunking function created successfully!


### Cell explanation

- The validation checks prevent invalid settings, such as an overlap greater than or equal to the chunk size.
- `start` tracks the first character of the current chunk.
- `end` is the position immediately after the last character selected by Python slicing.
- `text[start:end]` extracts the chunk.
- `start = end - overlap` moves to the next chunk while retaining the overlap.

**Expected output:** `Chunking function created successfully!`

## 7. Split all documents into chunks

Now we apply the function to every loaded document. Each chunk stores:
- `source`: original filename;
- `chunk_id`: its number within that document;
- `text`: the chunk content.

Keeping this metadata makes source citations possible later.

In [6]:
all_chunks = []

for doc in loaded_documents:
    chunks = split_text(doc["content"], chunk_size=100, overlap=20)

    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "source": doc["source"],
            "chunk_id": i + 1,
            "text": chunk
        })

print("Total chunks created:", len(all_chunks))

Total chunks created: 15


### Cell explanation and output analysis

- We process each document separately.
- `enumerate()` gives each chunk an index.
- `i + 1` makes chunk numbering start at 1 instead of 0.
- Every chunk keeps its source filename.

**Expected output:** The total number of chunks. It may be more than five because each original document can produce multiple chunks.

## 8. Inspect the chunks

Before building retrieval, inspect what the chunking function produced. This helps us notice chunk boundaries and the repeated overlap.

In [7]:
for chunk in all_chunks:
    print("\nSource:", chunk["source"])
    print("Chunk ID:", chunk["chunk_id"])
    print("Text:", repr(chunk["text"]))
    print("-" * 60)


Source: python.txt
Chunk ID: 1
Text: 'Python is a high-level programming language.\n    It is easy to learn and widely used in data science'
------------------------------------------------------------

Source: python.txt
Chunk ID: 2
Text: 'used in data science,\n    artificial intelligence, web development and automation.\n    Python suppor'
------------------------------------------------------------

Source: python.txt
Chunk ID: 3
Text: 'n.\n    Python supports object-oriented and procedural programming.'
------------------------------------------------------------

Source: java.txt
Chunk ID: 1
Text: 'Java is a popular object-oriented programming language.\n    It is widely used for enterprise applica'
------------------------------------------------------------

Source: java.txt
Chunk ID: 2
Text: 'r enterprise applications,\n    Android development and backend systems.\n    Java runs on the Java Vi'
------------------------------------------------------------

Source: java.txt
Chu

### Output analysis

You should see multiple chunks for each source document. Some chunks may begin or end in the middle of a word because the current method uses fixed character positions. `repr()` makes spaces and newline characters easier to notice.

This is an expected limitation of our simple character-based chunking strategy.

## 9. Convert chunks into TF-IDF vectors

A computer needs numerical representations to compare text using mathematical operations.

**TF-IDF** means **Term Frequency–Inverse Document Frequency**:
- Term Frequency (TF) measures how often a term appears in a text.
- Inverse Document Frequency (IDF) gives less weight to terms that appear in many texts and more weight to distinctive terms.
- TF-IDF combines these ideas into a numerical weight for each term.

`fit_transform()` learns the vocabulary and IDF weights from our chunks, then converts each chunk into a vector.

In [8]:
chunk_texts = [chunk["text"] for chunk in all_chunks]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(chunk_texts)

print("Number of chunks:", len(chunk_texts))
print("TF-IDF matrix shape:", tfidf_matrix.shape)

Number of chunks: 15
TF-IDF matrix shape: (15, 87)


### How to understand the matrix shape

If the output shape is `(15, 60)`, for example:
- 15 rows represent 15 chunks.
- 60 columns represent 60 distinct vocabulary terms learned from the chunks.

Your exact numbers may differ if the document text or chunk settings change.

Each row is a numerical representation of one chunk.

## 10. Inspect the TF-IDF vocabulary

The vocabulary is the set of terms the vectorizer learned from all chunks. These terms correspond to columns in the TF-IDF matrix.

In [9]:
print("Vocabulary:")
print(vectorizer.get_feature_names_out())

Vocabulary:
['allows' 'and' 'android' 'applica' 'applications' 'artificial' 'as' 'ate'
 'automation' 'backend' 'branch' 'commands' 'computers' 'data' 'database'
 'deep' 'delete' 'development' 'easy' 'enterprise' 'finds' 'for' 'from'
 'high' 'image' 'in' 'insert' 'intelligence' 'is' 'it' 'java' 'jvm'
 'labelled' 'language' 'layers' 'learn' 'learning' 'level' 'machine'
 'management' 'multiple' 'natural' 'networks' 'neural' 'object' 'of' 'on'
 'oriented' 'patterns' 'popular' 'procedural' 'processing' 'programming'
 'python' 'query' 'recognition' 'relational' 'retrieve' 'rts' 'runs'
 'science' 'select' 'speech' 'sql' 'stands' 'store' 'structured' 'subset'
 'such' 'supervised' 'suppor' 'supports' 'systems' 'the' 'to' 'training'
 'unlabelled' 'unsupervised' 'update' 'used' 'uses' 'vi' 'virtual' 'web'
 'widely' 'with' 'works']


### Output analysis

The output is an array of unique terms found in the documents after the vectorizer's text processing. Very common words may receive low IDF weights. The vocabulary size should match the number of columns in `tfidf_matrix`.

## 11. Calculate cosine similarity for one question

**Cosine similarity** compares the direction of two vectors. For TF-IDF vectors, a higher score generally means the question and chunk share more informative terms.

For this project, scores are usually between 0 and 1:
- closer to 1: stronger keyword similarity;
- closer to 0: little or no shared weighted vocabulary.

This is not the same as true understanding. TF-IDF is keyword-based, so a Java passage may score for a Python question if both contain words such as “programming language”.

In [10]:
question = "What is Python programming?"

question_vector = vectorizer.transform([question])

similarity_scores = cosine_similarity(
    question_vector,
    tfidf_matrix
)[0]

print("Similarity scores:")
print(similarity_scores)

Similarity scores:
[0.47039309 0.17140153 0.44338792 0.33605931 0.         0.
 0.0865562  0.         0.         0.09638347 0.         0.
 0.08759396 0.08662612 0.        ]


### Cell explanation

- `transform()` converts the new question using the vocabulary and IDF weights already learned from the chunks. We use `transform`, not `fit_transform`, because we must not relearn the vocabulary from each question.
- `cosine_similarity()` compares the question vector with every chunk vector.
- The result contains one score per chunk.

A score of `0.0` means the question and chunk have no overlapping weighted terms in this TF-IDF representation.

## 12. Display every chunk's similarity score

This inspection cell shows which chunks receive higher scores for the test question.

In [11]:
for i, score in enumerate(similarity_scores):
    print("\nChunk:", i + 1)
    print("Source:", all_chunks[i]["source"])
    print("Chunk ID:", all_chunks[i]["chunk_id"])
    print("Similarity:", round(float(score), 3))
    print("Text:", all_chunks[i]["text"])
    print("-" * 60)


Chunk: 1
Source: python.txt
Chunk ID: 1
Similarity: 0.47
Text: Python is a high-level programming language.
    It is easy to learn and widely used in data science
------------------------------------------------------------

Chunk: 2
Source: python.txt
Chunk ID: 2
Similarity: 0.171
Text: used in data science,
    artificial intelligence, web development and automation.
    Python suppor
------------------------------------------------------------

Chunk: 3
Source: python.txt
Chunk ID: 3
Similarity: 0.443
Text: n.
    Python supports object-oriented and procedural programming.
------------------------------------------------------------

Chunk: 4
Source: java.txt
Chunk ID: 1
Similarity: 0.336
Text: Java is a popular object-oriented programming language.
    It is widely used for enterprise applica
------------------------------------------------------------

Chunk: 5
Source: java.txt
Chunk ID: 2
Similarity: 0.0
Text: r enterprise applications,
    Android development and backend syste

### Output analysis

Look for the highest scores. For the question “What is Python programming?”, Python chunks should generally rank highly. A Java chunk may also receive a score because it contains the word “programming”.

This illustrates an important limitation: **similarity is a ranking signal, not proof that a passage answers the question.**

## 13. Retrieve the top relevant chunks

Top-K retrieval means selecting the K chunks with the highest similarity scores. Here, K is 3.

We also ignore chunks whose similarity is exactly zero. This avoids passing completely unrelated chunks to Gemini.

In [12]:
top_k = 3

top_indices = np.argsort(similarity_scores)[::-1][:top_k]

retrieved_chunks = []

for index in top_indices:
    if similarity_scores[index] > 0:
        retrieved_chunks.append(all_chunks[index])

print("Top retrieved chunks:\n")

for rank, chunk in enumerate(retrieved_chunks, start=1):
    index = all_chunks.index(chunk)
    print("Rank:", rank)
    print("Source:", chunk["source"])
    print("Chunk ID:", chunk["chunk_id"])
    print("Similarity:", round(float(similarity_scores[index]), 3))
    print("Text:", chunk["text"])
    print("-" * 60)

Top retrieved chunks:

Rank: 1
Source: python.txt
Chunk ID: 1
Similarity: 0.47
Text: Python is a high-level programming language.
    It is easy to learn and widely used in data science
------------------------------------------------------------
Rank: 2
Source: python.txt
Chunk ID: 3
Similarity: 0.443
Text: n.
    Python supports object-oriented and procedural programming.
------------------------------------------------------------
Rank: 3
Source: java.txt
Chunk ID: 1
Similarity: 0.336
Text: Java is a popular object-oriented programming language.
    It is widely used for enterprise applica
------------------------------------------------------------


### Output analysis

The output should list up to three chunks, ordered from highest to lowest similarity. There may be fewer than three if fewer than three chunks have a score greater than zero.

We retain the chunk's source and ID so that the final answer can point back to the retrieved passage.

## 14. Connect to Gemini securely

We use a Google Colab Secret rather than writing an API key directly in the notebook.

### Set up the secret

1. Open the **Secrets** (key icon) panel in Google Colab.
2. Create a secret named `GOOGLE_API_KEY`.
3. Paste your Gemini API key as its value.
4. Enable notebook access for that secret.

Do not share a notebook containing your API key. The notebook contains only the secret name, not the key itself.

In [13]:
from google.colab import userdata

api_key = userdata.get("GOOGLE_API_KEY")

if not api_key:
    raise ValueError(
        "GOOGLE_API_KEY was not found. Add it in Colab Secrets "
        "and enable notebook access."
    )

client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully!")

Gemini client initialized successfully!


### Cell explanation

- `userdata.get()` reads the secret from Colab.
- The `if not api_key` check gives a clear error if the secret is missing.
- `genai.Client()` creates the Gemini API client.

**Expected output:** `Gemini client initialized successfully!`

## 15. Discover and test available Gemini models

Model availability and capacity can change. Instead of assuming that one fixed model name will work, this cell lists models visible to your API key, keeps models that support `generateContent`, and tests candidates until one returns a response.

The first successfully tested model is stored in `working_model` and used in the rest of the notebook.

A model being listed does not guarantee that it has available capacity at every moment. Temporary overload, quota, or rate-limit errors can still occur.

In [14]:
models_url = "https://generativelanguage.googleapis.com/v1beta/models"

model_response = requests.get(
    models_url,
    params={"key": api_key},
    timeout=30
)
model_response.raise_for_status()

available_models = model_response.json().get("models", [])

generation_models = [
    model for model in available_models
    if "generateContent" in model.get("supportedGenerationMethods", [])
]

print("Models supporting generateContent:", len(generation_models))

# Prefer Flash/Lite models for this small text-generation task.
generation_models.sort(
    key=lambda model: (
        "flash" not in model.get("name", "").lower(),
        "lite" not in model.get("name", "").lower()
    )
)

working_model = None

for model in generation_models[:10]:
    model_name = model["name"].replace("models/", "")
    print("\nTesting:", model_name)

    try:
        test_response = client.models.generate_content(
            model=model_name,
            contents="Reply with exactly: Gemini connection successful."
        )

        if test_response.text:
            working_model = model_name
            print("SUCCESS! Working model:", working_model)
            print("Response:", test_response.text)
            break

    except Exception as error:
        print("This model did not respond:", str(error)[:300])
        time.sleep(2)

if working_model is None:
    raise RuntimeError(
        "No tested model responded. Check API key permissions, "
        "quota/rate limits, and temporary service availability."
    )

Models supporting generateContent: 49

Testing: gemini-flash-lite-latest
SUCCESS! Working model: gemini-flash-lite-latest
Response: Gemini connection successful.


### Cell explanation and output analysis

- The REST API request lists models available to the API key.
- `supportedGenerationMethods` is checked for `generateContent`.
- Flash/Lite models are tried first because this project needs simple text generation.
- Each candidate is tested with a short prompt.
- The first successful model name is saved in `working_model`.

In the user's earlier test, the successful model was `gemini-flash-lite-latest`. This cell discovers and tests candidates again when the notebook is run; the selected model may differ over time.

If no model works, check the API key, quota/rate limits, and Google's service status. Model discovery cannot bypass an exhausted quota or temporary overload.

## 16. Test Gemini directly

Before connecting Gemini to retrieval, send one simple prompt. This separates API connection problems from RAG logic problems.

In [15]:
test_response = client.models.generate_content(
    model=working_model,
    contents="Explain Python in two simple sentences."
)

print("Model used:", working_model)
print("\nGemini response:")
print(test_response.text)

Model used: gemini-flash-lite-latest

Gemini response:
Python is a popular programming language known for its clear, readable syntax that resembles plain English. It is widely used for web development, data analysis, artificial intelligence, and automating everyday tasks.


### Output analysis

Gemini should return a short explanation of Python. The exact wording can vary. If this cell works, the API connection and selected model are responding successfully.

## 17. Create the RAG prompt

The prompt combines:
- the user's question;
- the retrieved passages;
- instructions to answer only from those passages.

Each passage includes a source filename and chunk ID. Gemini is instructed to cite only the supplied source labels and to say when the answer is not present.

In [16]:
def create_prompt(question, retrieved_chunks):
    context_parts = []

    for chunk in retrieved_chunks:
        source_label = f"{chunk['source']} | chunk {chunk['chunk_id']}"
        context_parts.append(
            f"[{source_label}]\n{chunk['text']}"
        )

    context = "\n\n".join(context_parts)

    prompt = f'''
You are a question-answering assistant that must use only the context below.

Rules:
1. Answer only with information supported by the context.
2. Do not use outside knowledge or make up facts.
3. If the context does not contain the answer, say exactly:
   "I don't have enough information in the provided documents."
4. When you answer, include source labels exactly as shown in square brackets.
5. Do not invent source filenames or chunk IDs.
6. Keep the answer clear and concise.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
'''
    return prompt


print("RAG prompt function created successfully!")

RAG prompt function created successfully!


### Cell explanation

- We build one context section from the retrieved chunks.
- Each passage receives a source label such as `[python.txt | chunk 1]`.
- The instructions tell Gemini to use only this context and admit when the answer is missing.
- The function returns one complete prompt string.

A prompt instruction helps guide the model, but it is not a mathematical guarantee that a model can never make a mistake. We will also display the retrieved passages so the answer can be checked.

## 18. Build the complete RAG assistant

This function performs the full pipeline for one question:
1. Convert the question to a TF-IDF vector.
2. Calculate similarity against all chunks.
3. Retrieve the top-K chunks with positive scores.
4. Return an honest gap if no chunks match.
5. Send the question and retrieved context to Gemini.
6. Return the answer, sources, scores, and retrieved passages.

We use a small minimum similarity threshold. It helps reject questions with only extremely weak keyword matches. This threshold is a simple teaching choice, not a universally optimal value.

In [17]:
MIN_SIMILARITY = 0.05

def rag_assistant(question, top_k=3):
    question_vector = vectorizer.transform([question])

    scores = cosine_similarity(
        question_vector,
        tfidf_matrix
    )[0]

    ranked_indices = np.argsort(scores)[::-1]

    retrieved_chunks = []

    for index in ranked_indices:
        if scores[index] >= MIN_SIMILARITY:
            chunk_record = all_chunks[index].copy()
            chunk_record["similarity"] = float(scores[index])
            retrieved_chunks.append(chunk_record)

        if len(retrieved_chunks) == top_k:
            break

    # If nothing passes the threshold, do not ask Gemini to guess.
    if not retrieved_chunks:
        return {
            "question": question,
            "answer": "I don't have enough information in the provided documents.",
            "sources": [],
            "retrieved_chunks": []
        }

    prompt = create_prompt(question, retrieved_chunks)

    response = client.models.generate_content(
        model=working_model,
        contents=prompt
    )

    sources = list(dict.fromkeys(
        chunk["source"] for chunk in retrieved_chunks
    ))

    return {
        "question": question,
        "answer": response.text or "The model returned an empty response.",
        "sources": sources,
        "retrieved_chunks": retrieved_chunks
    }


print("RAG assistant is ready!")

RAG assistant is ready!


### Cell explanation and output analysis

- `MIN_SIMILARITY` is the minimum keyword-similarity score required for a chunk to be considered.
- `ranked_indices` orders chunks from highest score to lowest.
- We copy each chunk and attach its similarity score.
- If no chunk passes the threshold, the function returns an honest gap without calling Gemini.
- Otherwise, it builds the prompt and calls the selected Gemini model.
- `sources` lists the filenames of the passages supplied to Gemini.

**Important limitation:** A question can have keyword overlap but still not be answerable from the retrieved passage. Always inspect the passages and citations. TF-IDF can also miss relevant information when the question uses different wording from the document.

## 19. Ask the first question

This is the first end-to-end RAG test. The assistant should retrieve Python-related text and generate an answer based on that context.

In [18]:
question = "What is Python?"

result = rag_assistant(question)

print("QUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCE DOCUMENTS:")
for source in result["sources"]:
    print("-", source)

QUESTION:
What is Python?

ANSWER:
Python is a high-level programming language that is easy to learn and widely used in data science [python.txt | chunk 1]. It also supports object-oriented and procedural programming [python.txt | chunk 3].

SOURCE DOCUMENTS:
- python.txt
- java.txt


### Output analysis

A good answer should describe Python using details present in `python.txt`, such as that it is a high-level programming language and is used in data science, AI, web development, and automation.

The answer should include a source label. Wording may vary because Gemini generates the response.

## 20. Inspect the retrieved passages

A source filename alone is not enough to verify grounding. This cell shows the exact passages that were sent to Gemini, together with their chunk IDs and similarity scores.

In [19]:
print("RETRIEVED PASSAGES\n")

for rank, chunk in enumerate(result["retrieved_chunks"], start=1):
    print("Rank:", rank)
    print("Source:", chunk["source"])
    print("Chunk ID:", chunk["chunk_id"])
    print("Similarity:", round(chunk["similarity"], 3))
    print("Passage:", chunk["text"])
    print("-" * 70)

RETRIEVED PASSAGES

Rank: 1
Source: python.txt
Chunk ID: 1
Similarity: 0.411
Passage: Python is a high-level programming language.
    It is easy to learn and widely used in data science
----------------------------------------------------------------------
Rank: 2
Source: python.txt
Chunk ID: 3
Similarity: 0.283
Passage: n.
    Python supports object-oriented and procedural programming.
----------------------------------------------------------------------
Rank: 3
Source: java.txt
Chunk ID: 1
Similarity: 0.232
Passage: Java is a popular object-oriented programming language.
    It is widely used for enterprise applica
----------------------------------------------------------------------


### Output analysis

Check whether the passages actually support the answer. The filename and chunk ID identify where each passage came from.

The assistant's source list is based on the retrieved passages. Gemini's citation text is generated, so compare its labels with this retrieved-passage list rather than assuming a citation is correct automatically.

## 21. Ask questions dynamically

Change the `question` value and rerun the cell to ask something else. The RAG function repeats retrieval and generation for each new question.

In [20]:
question = "What is Java used for?"

result = rag_assistant(question)

print("QUESTION:", result["question"])
print("\nANSWER:")
print(result["answer"])

print("\nRETRIEVED SOURCES:")
for chunk in result["retrieved_chunks"]:
    print(
        f"- {chunk['source']} | chunk {chunk['chunk_id']} "
        f"| similarity {chunk['similarity']:.3f}"
    )

QUESTION: What is Java used for?

ANSWER:
Java is widely used for enterprise applications [java.txt | chunk 1, java.txt | chunk 2], Android development, and backend systems [java.txt | chunk 2].

RETRIEVED SOURCES:
- java.txt | chunk 1 | similarity 0.545
- sql.txt | chunk 1 | similarity 0.320
- java.txt | chunk 2 | similarity 0.247


### Output analysis

For this question, `java.txt` should normally be among the retrieved sources. The answer should be grounded in the text about enterprise applications, Android development, backend systems, and the JVM.

If another document appears in the results, inspect its similarity score and passage. Retrieval is based on matching terms, not perfect understanding.

## 22. Test an unsupported question

A useful RAG assistant should not invent an answer when its approved documents do not contain the information.

Our sample documents do not explain quantum computing. We test whether the retrieval stage finds any sufficiently similar passages.

In [21]:
question = "What is quantum computing?"

result = rag_assistant(question)

print("QUESTION:", result["question"])
print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")
if result["sources"]:
    for source in result["sources"]:
        print("-", source)
else:
    print("No relevant source passages found.")

QUESTION: What is quantum computing?

ANSWER:
I don't have enough information in the provided documents.

SOURCES:
- java.txt
- python.txt
- machine_learning.txt


### Expected behaviour and analysis

For this question, the expected response is:

`I don't have enough information in the provided documents.`

The sources list should be empty if no chunk reaches the similarity threshold.

This unsupported-question check is based on retrieval. It is a simple safeguard, not a perfect guarantee: an unrelated question that shares words with a document could still retrieve a passage. Inspecting the retrieved context remains important.

## 23. Evaluate the RAG assistant with five questions

The assignment asks us to evaluate the assistant using five questions. We use four questions covered by our documents and one unsupported question.

Evaluation checks:
- whether the expected source is retrieved;
- whether the assistant returns an answer or an honest gap;
- what sources were retrieved.

This is a small manual evaluation, not a comprehensive measure of answer quality.

In [22]:
evaluation_questions = [
    {
        "question": "What is Python?",
        "expected_source": "python.txt",
        "should_be_supported": True
    },
    {
        "question": "What is Java used for?",
        "expected_source": "java.txt",
        "should_be_supported": True
    },
    {
        "question": "What does SQL stand for?",
        "expected_source": "sql.txt",
        "should_be_supported": True
    },
    {
        "question": "What does supervised learning use?",
        "expected_source": "machine_learning.txt",
        "should_be_supported": True
    },
    {
        "question": "What is quantum computing?",
        "expected_source": None,
        "should_be_supported": False
    }
]

evaluation_results = []

for item in evaluation_questions:
    result = rag_assistant(item["question"])

    retrieved_sources = result["sources"]
    expected_source = item["expected_source"]

    if item["should_be_supported"]:
        passed = expected_source in retrieved_sources
    else:
        passed = len(retrieved_sources) == 0

    evaluation_results.append({
        "question": item["question"],
        "expected_source": expected_source or "No source expected",
        "retrieved_sources": ", ".join(retrieved_sources) if retrieved_sources else "None",
        "passed_source_check": passed,
        "answer": result["answer"]
    })

for i, item in enumerate(evaluation_results, start=1):
    print(f"\nTEST {i}")
    print("Question:", item["question"])
    print("Expected source:", item["expected_source"])
    print("Retrieved sources:", item["retrieved_sources"])
    print("Source check:", "PASS" if item["passed_source_check"] else "CHECK")
    print("Answer:", item["answer"])
    print("-" * 75)

passed_count = sum(
    item["passed_source_check"] for item in evaluation_results
)

print("\nEVALUATION SUMMARY")
print("Passed source checks:", passed_count, "/", len(evaluation_results))
print("Source retrieval pass rate:", round(
    passed_count / len(evaluation_results) * 100, 2
), "%")


TEST 1
Question: What is Python?
Expected source: python.txt
Retrieved sources: python.txt, java.txt
Source check: PASS
Answer: Python is a high-level programming language that is easy to learn and widely used in data science [python.txt | chunk 1]. It supports object-oriented and procedural programming [python.txt | chunk 3].
---------------------------------------------------------------------------

TEST 2
Question: What is Java used for?
Expected source: java.txt
Retrieved sources: java.txt, sql.txt
Source check: PASS
Answer: Based on the provided documents, Java is widely used for enterprise applications [java.txt | chunk 1, java.txt | chunk 2], Android development, and backend systems [java.txt | chunk 2].
---------------------------------------------------------------------------

TEST 3
Question: What does SQL stand for?
Expected source: sql.txt
Retrieved sources: sql.txt, java.txt
Source check: PASS
Answer: SQL stands for Structured Query Language [sql.txt | chunk 1].
-------

### How to interpret the evaluation

- A supported question passes the source check when its expected document is among the retrieved sources.
- The unsupported question passes when no source is retrieved.
- `PASS` evaluates source retrieval behaviour only. It does not prove that every generated sentence is factually correct.
- Read the generated answers and compare them with the original document text.

If a test shows `CHECK`, inspect the retrieved chunks and similarity threshold. Do not simply change the expected result to make the test pass.

## 24. Final project summary

### What we built

| Component | What it does |
|---|---|
| Knowledge base | Stores the five sample documents |
| Document loading | Reads document text and preserves filenames |
| Chunking | Splits each document into smaller character-based pieces |
| TF-IDF | Converts chunks and questions into numerical vectors |
| Cosine similarity | Scores question-to-chunk similarity |
| Top-K retrieval | Selects the highest-scoring chunks |
| Gemini | Generates an answer using retrieved context |
| Source metadata | Keeps source filenames and chunk IDs |
| Unsupported-question handling | Returns an honest gap when no chunk passes the threshold |
| Evaluation | Tests five questions |

### Limitations of this beginner implementation

1. TF-IDF is keyword-based and may miss relevant passages expressed with different words.
2. Character-based chunking can split words or sentences.
3. The similarity threshold is a simple heuristic and may need adjustment for another dataset.
4. Gemini may still produce an unsupported statement despite prompt instructions. Check its answer against the retrieved passages.
5. The sample knowledge base is small and contains only five short documents.

